# 03. Public stage model comparison

`configs/models.py`의 13개 model, network, head 조합을 public dataset stage에서 학습,
평가한 결과를 비교한다. 새로운 학습이나 평가는 실행하지 않고
`outputs/metrics_summary.csv`와 각 run의 `history.json`만 읽는다. stage 간 비교는
다루지 않는다.

이 notebook은 agent가 실행하지 않는다. 모든 code cell은 사용자가 Jupyter에서 직접 실행한다.

## 0. 환경 설정

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils.plot import show_history

In [ ]:
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs", "public")
SUMMARY_CSV = os.path.join(OUTPUTS_DIR, "metrics_summary.csv")
STAGE = "public"

print("outputs dir: ", OUTPUTS_DIR)
print("summary csv:", SUMMARY_CSV)
print("stage:       ", STAGE)

## 1. metrics_summary.csv 로딩

`scripts/collect_metrics.py`가 이미 만들어 둔 `outputs/metrics_summary.csv`를 읽고
`STAGE`에 해당하는 13개 row만 사용한다.

In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
summary = summary[summary["dataset"] == STAGE].reset_index(drop=True)

print("rows:", len(summary), "cols:", len(summary.columns))
print(list(summary.columns))
summary.head()

## 2. 성능 표

test `iou` 내림차순으로 정렬한 표를 출력한다.

In [ ]:
display_cols = ["model", "network", "head", "iou", "mcd", "maxcd", "pck_002", "pck_005", "sr"]

stage_rows = summary.sort_values("iou", ascending=False)
display(stage_rows[display_cols].reset_index(drop=True))

## 3. Family별 색상, 정렬 유틸

`slides/assets/make_result_figs.py`와 동일한 이름의 상수와 함수를 notebook에 재정의한다.
model을 `coordinate`, `mask`, `dense`, `box` 4개 family로 묶어 이후 그래프에서 같은
색으로 표시한다.

In [ ]:
FAMILY_OF_MODEL = {
    "reg": "coordinate", "offset": "coordinate", "gcn": "coordinate",
    "seg": "mask", "hybrid": "mask", "torchseg": "mask",
    "peak": "dense", "ridge": "dense",
    "det": "box", "torchdet": "box", "yolo": "box", "detr": "box",
}
FAMILY_ORDER = ["coordinate", "mask", "dense", "box"]
FAMILY_COLOR = {
    "coordinate": "#1f77b4", "mask": "#2ca02c", "dense": "#d62728", "box": "#9467bd",
}
LINE_STYLES = ["-", "--", "-.", ":"]


def run_label(row):
    """Return a short run label combining model and network."""
    return "%s/%s" % (row["model"], row["network"])


def sorted_runs(rows):
    """Return rows ordered by family then descending test iou."""
    rows = rows.copy()
    rows["family_order"] = rows["model"].map(FAMILY_OF_MODEL).map(FAMILY_ORDER.index)
    rows = rows.sort_values(["family_order", "iou"], ascending=[True, False])
    return rows.drop(columns="family_order")


def style_of(index_in_family):
    """Return the line style used to separate runs inside one family."""
    return LINE_STYLES[index_in_family % len(LINE_STYLES)]

## 4. Test metric bar chart

`iou`와 `pck_002`를 수평 막대 그래프로 비교한다. 색상은 family 기준이다.

In [ ]:
def plot_metric_stage(rows):
    """Draw one stage test metric bar chart with iou and pck_002 side by side."""
    runs = rows.sort_values("iou")
    labels = [run_label(r) for _, r in runs.iterrows()]
    colors = [FAMILY_COLOR[FAMILY_OF_MODEL[m]] for m in runs["model"]]
    pos = np.arange(len(runs))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, key, name in zip(axes, ["iou", "pck_002"], ["test IoU", "test PCK@0.02"]):
        values = runs[key].to_numpy(dtype=float)
        ax.barh(pos, values, color=colors, height=0.65)
        ax.set_yticks(pos)
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlim(0.0, 1.0)
        ax.set_xlabel(name)
        ax.grid(True, axis="x", alpha=0.3)
        for y, value in zip(pos, values):
            ax.text(min(value + 0.01, 0.98), y, "%.3f" % value, va="center", fontsize=7)
    axes[1].set_yticklabels([])
    fig.suptitle("%s stage test metrics" % STAGE, fontsize=12)
    fig.tight_layout()
    plt.show()


plot_metric_stage(summary)

## 5. 학습 곡선

각 run의 `history.json`에서 `valid.iou` 곡선을 읽어 하나의 axes에 family별 색상으로
표시한다. 점은 각 run의 best epoch(최대 valid iou)를 나타낸다.

In [ ]:
def read_valid_iou(row):
    """Return the validation iou curve of one run, or None when unavailable."""
    path = os.path.join(OUTPUTS_DIR, row["model"], row["exp_name"], "history.json")
    if not os.path.isfile(path):
        return None
    with open(path, encoding="utf-8") as f:
        history = json.load(f)
    curve = history.get("valid", {}).get("iou")
    if not curve:
        return None
    return np.asarray(curve, dtype=float)


def draw_history(ax, runs, title):
    """Draw validation iou curves of one stage on the given axes."""
    seen = {}
    for _, row in runs.iterrows():
        curve = read_valid_iou(row)
        if curve is None:
            continue
        family = FAMILY_OF_MODEL[row["model"]]
        order = seen.get(family, 0)
        seen[family] = order + 1
        epochs = np.arange(1, len(curve) + 1)
        ax.plot(epochs, curve, color=FAMILY_COLOR[family], linestyle=style_of(order),
                linewidth=1.4, label=run_label(row))
        best = int(np.argmax(curve))
        ax.plot([best + 1], [curve[best]], marker="o", markersize=4,
                color=FAMILY_COLOR[family], linestyle="none")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("epoch")
    ax.set_ylabel("valid IoU")
    ax.set_ylim(0.6, 1.0)
    ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
    ax.grid(True, alpha=0.3)


fig, ax = plt.subplots(figsize=(12, 5))
draw_history(ax, sorted_runs(summary), "%s stage validation IoU" % STAGE)
ax.legend(fontsize=10, loc="lower right")
fig.tight_layout()
plt.show()

## 6. 개별 run 학습 곡선 상세

관심 있는 run 하나를 골라 `show_history`로 loss와 iou를 포함한 모든 history key를
함께 표시한다. `RUN_MODEL`, `RUN_EXP_NAME`을 바꾸면 다른 run을 확인할 수 있다.

In [ ]:
RUN_MODEL = "reg"
RUN_EXP_NAME = "reg_resnet50_spatial_public"

run_history_path = os.path.join(OUTPUTS_DIR, RUN_MODEL, RUN_EXP_NAME, "history.json")

with open(run_history_path, encoding="utf-8") as f:
    run_history = json.load(f)

show_history(run_history, title=RUN_EXP_NAME)